In [2]:
import pandas as pd
import numpy as np
import os

# =========================
# Load cleaned files
# =========================

train_path = "../MLOpsedian/data/processed/train_cleaned.csv"
test_path = "../MLOpsedian/data/processed/test_cleaned.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Original train shape:", train.shape)
print("Original test shape:", test.shape)

print("\nOriginal missing values:")
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())

print("\nRows with any missing in train:", train.isnull().any(axis=1).sum())
print("Rows without missing in train:", (~train.isnull().any(axis=1)).sum())


# =========================
# VERSION 1: Drop missing rows only
# =========================

train_dropmissing = train.dropna().copy()
test_dropmissing = test.copy()

print("\n==============================")
print("VERSION 1: DROP MISSING ONLY")
print("==============================")
print("Train shape:", train_dropmissing.shape)
print("Test shape:", test_dropmissing.shape)
print("Train missing:", train_dropmissing.isnull().sum().sum())
print("Test missing:", test_dropmissing.isnull().sum().sum())
print("Duplicate train IDs:", train_dropmissing[id_col].duplicated().sum())
print("Duplicate test IDs:", test_dropmissing[id_col].duplicated().sum())
print("Train-only columns:", set(train_dropmissing.columns) - set(test_dropmissing.columns))
print("Test-only columns:", set(test_dropmissing.columns) - set(train_dropmissing.columns))


# =========================
# VERSION 2: Drop missing + invalid rows
# =========================

invalid_rules = {
    "distance_to_river_m": {"min": 0, "max": None},
    "rainfall_7d_mm": {"min": 0, "max": None},
    "monthly_rainfall_mm": {"min": 0, "max": None},
    "built_up_percent": {"min": 0, "max": 100},
    "drainage_index": {"min": 0, "max": 1},
    "ndvi": {"min": -1, "max": 1},
    "ndwi": {"min": -1, "max": 1},
    "elevation_m": {"min": 0, "max": None}
}

missing_mask = train.isnull().any(axis=1)
invalid_mask = pd.Series(False, index=train.index)

invalid_report = []

for col, rule in invalid_rules.items():
    if col in train.columns:
        col_invalid = pd.Series(False, index=train.index)
        
        if rule["min"] is not None:
            col_invalid |= train[col] < rule["min"]
        
        if rule["max"] is not None:
            col_invalid |= train[col] > rule["max"]
        
        col_invalid = col_invalid.fillna(False)
        invalid_mask |= col_invalid
        
        invalid_report.append({
            "column": col,
            "invalid_count": int(col_invalid.sum()),
            "min_value": train[col].min(),
            "max_value": train[col].max()
        })

invalid_report_df = pd.DataFrame(invalid_report)

train_dropmissing_invalid = train.loc[~(missing_mask | invalid_mask)].copy()
test_dropmissing_invalid = test.copy()

print("\n==============================")
print("VERSION 2: DROP MISSING + INVALID")
print("==============================")
print("Rows with missing:", int(missing_mask.sum()))
print("Rows with invalid values:", int(invalid_mask.sum()))
print("Rows with missing OR invalid:", int((missing_mask | invalid_mask).sum()))
print("Rows kept:", train_dropmissing_invalid.shape[0])

print("\nTrain shape:", train_dropmissing_invalid.shape)
print("Test shape:", test_dropmissing_invalid.shape)
print("Train missing:", train_dropmissing_invalid.isnull().sum().sum())
print("Test missing:", test_dropmissing_invalid.isnull().sum().sum())
print("Duplicate train IDs:", train_dropmissing_invalid[id_col].duplicated().sum())
print("Duplicate test IDs:", test_dropmissing_invalid[id_col].duplicated().sum())
print("Train-only columns:", set(train_dropmissing_invalid.columns) - set(test_dropmissing_invalid.columns))
print("Test-only columns:", set(test_dropmissing_invalid.columns) - set(train_dropmissing_invalid.columns))

print("\nInvalid value report:")
display(invalid_report_df)


# =========================
# Check target balance
# =========================

def target_balance(df, name):
    bins = pd.cut(
        df[target],
        bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
        labels=[
            "0.0-0.2 very low",
            "0.2-0.4 low",
            "0.4-0.6 medium",
            "0.6-0.8 high",
            "0.8-1.0 very high"
        ]
    )
    
    report = pd.DataFrame({
        "count": bins.value_counts().sort_index(),
        "percentage": (bins.value_counts(normalize=True).sort_index() * 100).round(2)
    })
    
    print("\n==============================")
    print(name)
    print("==============================")
    print("Shape:", df.shape)
    print(report)
    
    return report

balance_full = target_balance(train, "Original cleaned train")
balance_dropmissing = target_balance(train_dropmissing, "Drop missing train")
balance_strict = target_balance(train_dropmissing_invalid, "Drop missing + invalid train")


# =========================
# Save files
# =========================

os.makedirs("../MLOpsedian/data/processed", exist_ok=True)
os.makedirs("../MLOpsedian/reports", exist_ok=True)

train_dropmissing.to_csv("../MLOpsedian/data/processed/train_dropmissing.csv", index=False)
test_dropmissing.to_csv("../MLOpsedian/data/processed/test_dropmissing.csv", index=False)

train_dropmissing_invalid.to_csv("../MLOpsedian/data/processed/train_dropmissing_invalid.csv", index=False)
test_dropmissing_invalid.to_csv("../MLOpsedian/data/processed/test_dropmissing_invalid.csv", index=False)

invalid_report_df.to_csv("../MLOpsedian/reports/invalid_rows_removed_report.csv", index=False)
balance_full.to_csv("../MLOpsedian/reports/balance_original_cleaned_train.csv")
balance_dropmissing.to_csv("../MLOpsedian/reports/balance_dropmissing_train.csv")
balance_strict.to_csv("../MLOpsedian/reports/balance_dropmissing_invalid_train.csv")

print("\n==============================")
print("FILES SAVED")
print("==============================")
print("../MLOpsedian/data/processed/train_dropmissing.csv")
print("../MLOpsedian/data/processed/test_dropmissing.csv")
print("../MLOpsedian/data/processed/train_dropmissing_invalid.csv")
print("../MLOpsedian/data/processed/test_dropmissing_invalid.csv")
print("../MLOpsedian/reports/invalid_rows_removed_report.csv")

Original train shape: (19700, 69)
Original test shape: (5300, 68)

Original missing values:
Train missing: 32761
Test missing: 0

Rows with any missing in train: 4182
Rows without missing in train: 15518

VERSION 1: DROP MISSING ONLY
Train shape: (15518, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()

VERSION 2: DROP MISSING + INVALID
Rows with missing: 4182
Rows with invalid values: 683
Rows with missing OR invalid: 4779
Rows kept: 14921

Train shape: (14921, 69)
Test shape: (5300, 68)
Train missing: 0
Test missing: 0
Duplicate train IDs: 0
Duplicate test IDs: 0
Train-only columns: {'flood_risk_score'}
Test-only columns: set()

Invalid value report:


,column,invalid_count,min_value,max_value
0,distance_to_river_m,117,-485.397750,16802.000000
1,rainfall_7d_mm,96,-49.199358,914.823318
2,monthly_rainfall_mm,96,-79.170571,2032.274155
3,built_up_percent,72,1.000000,178.263721
4,drainage_index,71,0.003000,1.779877
5,ndvi,71,-0.792000,1.595039
6,ndwi,71,-1.000000,1.596308
7,elevation_m,97,-79.564149,2148.000000



Original cleaned train
Shape: (19700, 69)
                   count  percentage
flood_risk_score                    
0.0-0.2 very low    2390       12.13
0.2-0.4 low         4402       22.35
0.4-0.6 medium      7650       38.83
0.6-0.8 high        3353       17.02
0.8-1.0 very high   1905        9.67

Drop missing train
Shape: (15518, 69)
                   count  percentage
flood_risk_score                    
0.0-0.2 very low    1899       12.24
0.2-0.4 low         3455       22.26
0.4-0.6 medium      5993       38.62
0.6-0.8 high        2655       17.11
0.8-1.0 very high   1516        9.77

Drop missing + invalid train
Shape: (14921, 69)
                   count  percentage
flood_risk_score                    
0.0-0.2 very low    1816       12.17
0.2-0.4 low         3338       22.37
0.4-0.6 medium      5757       38.58
0.6-0.8 high        2561       17.16
0.8-1.0 very high   1449        9.71

FILES SAVED
../MLOpsedian/data/processed/train_dropmissing.csv
../MLOpsedian/data/processed

In [ ]:
# ============================================================
# CatBoost Alpha Pack on Drop-Missing Dataset
# One-cell version
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# ============================================================
# 1. Load drop-missing datasets
# ============================================================

train_path = "../MLOpsedian/data/processed/train_dropmissing.csv"
test_path = "../MLOpsedian/data/processed/test_dropmissing.csv"

# Backup path in case your files are saved inside MLOpsedian folder


train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

id_col = "record_id"
target = "flood_risk_score"

print("Train path:", train_path)
print("Test path:", test_path)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("Target valid:", train[target].between(0, 1).all())

# ============================================================
# 2. Add Alpha Pack engineered features
# ============================================================

def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    # Use clipped columns if available
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    # 1. Golden distance-rainfall ratio
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    # 2. Flood spread ratio
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    # 3. Water velocity proxy
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    # 4. Water volume proxy
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("\nAfter feature engineering:")
print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)
print("Train missing:", train_fe.isnull().sum().sum())
print("Test missing:", test_fe.isnull().sum().sum())

# ============================================================
# 3. Define Alpha Pack features
# ============================================================

alpha_pack_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2"
]

missing_train_features = [col for col in alpha_pack_features if col not in train_fe.columns]
missing_test_features = [col for col in alpha_pack_features if col not in test_fe.columns]

print("\nFeature check:")
print("Number of alpha pack features:", len(alpha_pack_features))
print("Missing train features:", missing_train_features)
print("Missing test features:", missing_test_features)

if len(missing_train_features) > 0 or len(missing_test_features) > 0:
    raise ValueError("Some alpha pack features are missing. Stop and check columns.")

# ============================================================
# 4. Prepare X, y, X_test
# ============================================================

X = train_fe[alpha_pack_features].copy()
y = train_fe[target].copy()

X_test = test_fe[alpha_pack_features].copy()
test_ids = test_fe[id_col].copy()

print("\nModel data:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())
print("Feature columns match:", list(X.columns) == list(X_test.columns))

# ============================================================
# 5. Identify categorical features
# ============================================================

cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("\nCategorical features:")
print("Number of categorical features:", len(cat_features))
print(cat_features)

# ============================================================
# 6. Target bins for balance-aware StratifiedKFold
# ============================================================

target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("\nTarget bin distribution:")
print(target_bins.value_counts().sort_index())

# ============================================================
# 7. Submission validation function
# ============================================================

def validate_submission(submission):
    print("\nSubmission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

# ============================================================
# 8. Train CatBoost Alpha Pack on drop-missing data
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse_scores = []
mae_scores = []
r2_scores = []

test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    
    valid_preds = model.predict(X_valid)
    valid_preds = np.clip(valid_preds, 0, 1)
    
    rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    mae = mean_absolute_error(y_valid, valid_preds)
    r2 = r2_score(y_valid, valid_preds)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print("Fold RMSE:", rmse)
    print("Fold MAE :", mae)
    print("Fold R²  :", r2)
    
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / skf.n_splits

print("\n" + "=" * 70)
print("CatBoost Alpha Pack Drop-Missing CV Results")
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE :", np.mean(mae_scores))
print("Average R²  :", np.mean(r2_scores))

# ============================================================
# 9. Create submission
# ============================================================

dropmissing_preds = np.clip(test_preds, 0, 1)

print("\nPrediction distribution:")
print("Prediction min:", dropmissing_preds.min())
print("Prediction max:", dropmissing_preds.max())
print("Prediction mean:", dropmissing_preds.mean())
print("Prediction std:", dropmissing_preds.std())

sub_dropmissing = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": dropmissing_preds
})

validate_submission(sub_dropmissing)

os.makedirs("../submissions", exist_ok=True)

sub_dropmissing.to_csv(
    "../submissions/sub_v010_catboost_alpha_pack_dropmissing.csv",
    index=False
)

print("\nSaved: ../submissions/sub_v010_catboost_alpha_pack_dropmissing.csv")

# ============================================================
# 10. Compare with previous best
# ============================================================

comparison = pd.DataFrame([
    {
        "version": "sub_v008_catboost_alpha_pack",
        "dataset": "Full cleaned train",
        "rows": 19700,
        "cv_rmse": 0.23164778640435246,
        "cv_mae": 0.17673736118136243,
        "cv_r2": 0.03463008143352864,
        "public_score": 0.38308
    },
    {
        "version": "sub_v010_catboost_alpha_pack_dropmissing",
        "dataset": "Drop missing train",
        "rows": len(train),
        "cv_rmse": np.mean(rmse_scores),
        "cv_mae": np.mean(mae_scores),
        "cv_r2": np.mean(r2_scores),
        "public_score": None
    }
])

comparison["rmse_vs_previous"] = comparison["cv_rmse"] - comparison.loc[0, "cv_rmse"]
comparison["mae_vs_previous"] = comparison["cv_mae"] - comparison.loc[0, "cv_mae"]
comparison["r2_vs_previous"] = comparison["cv_r2"] - comparison.loc[0, "cv_r2"]

os.makedirs("../reports", exist_ok=True)

comparison.to_csv(
    "../reports/catboost_alpha_pack_dropmissing_comparison.csv",
    index=False
)

print("\nComparison with previous best:")
display(comparison)